In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D
from PIL import ImageFile
import pandas as pd
import numpy as np
import os

In [2]:
from tensorflow.keras import layers, models

In [ ]:
base_model = ResNet50(weights="imagenet", include_top=False)
base_model.trainable = False

model = models.Sequential()
model.add(base_model)
model.add(layers.GlobalAveragePooling2D())
model.summary()

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, None, None,     │    23,587,712 │
│                                 │ 2048)                  │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,587,712 (89.98 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 23,587,712 (89.98 MB)

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
ImageFile.LOAD_TRUNCATED_IMAGES = True
trainPath='/content/gdrive/MyDrive/train'
valiPath='/content/gdrive/MyDrive/val'

In [ ]:
trainImage = ImageDataGenerator(rescale=1./255).flow_from_directory(trainPath,target_size=(224,224),batch_size=32,class_mode='categorical')
valiImage = ImageDataGenerator(rescale=1./255).flow_from_directory(valiPath,target_size=(224,224),batch_size=32,class_mode='categorical')

Found 2872 images belonging to 18 classes.
Found 761 images belonging to 18 classes.


In [ ]:
trainLabel = []
valiLabel = []

for folder in sorted(os.listdir(trainPath)):
    folder_path = os.path.join(trainPath, folder)
    if os.path.isdir(folder_path):
        label = folder
        for file in sorted(os.listdir(folder_path)):
            if os.path.isfile(os.path.join(folder_path, file)):
                trainLabel.append((label, label+"/"+file))

for folder in sorted(os.listdir(valiPath)):
    folder_path = os.path.join(valiPath, folder)
    if os.path.isdir(folder_path):
        label = folder
        for file in sorted(os.listdir(folder_path)):
            if os.path.isfile(os.path.join(folder_path, file)):
                valiLabel.append((label, label+"/"+file))


In [ ]:
train = pd.DataFrame(trainLabel, columns=['label', 'pathname'])
vali = pd.DataFrame(valiLabel, columns=['label', 'pathname'])

train.to_csv('trainLabel.csv', index=False)
vali.to_csv('valiLabel.csv', index=False)

In [ ]:
train_features = model.predict(trainImage, verbose=1)
val_features = model.predict(valiImage, verbose=1)

train_filenames = trainImage.filenames
val_filenames = valiImage.filenames

df_train = pd.DataFrame(train_features)
df_val = pd.DataFrame(val_features)

df_train.insert(0, "filename", train_filenames)
df_val.insert(0, "filename", val_filenames)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


90/90 ━━━━━━━━━━━━━━━━━━━━ 1834s 21s/step
24/24 ━━━━━━━━━━━━━━━━━━━━ 467s 20s/step


In [ ]:
df_train.to_csv("trainFeature.csv", index=False)
df_val.to_csv("valiFeature.csv", index=False)

In [ ]:
import joblib

filename = 'restnet50.sav'
joblib.dump(model, filename)

['restnet50.sav']

# End of restnet50

In [3]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [102]:
trainFeature= pd.read_csv('/content/gdrive/MyDrive/ml/restnet50/featureCSV/trainFeature.csv')
valiFeature=pd.read_csv('/content/gdrive/MyDrive/ml/restnet50/featureCSV/valiFeature.csv')

trainFilename=trainFeature['filename']
valiFilename=valiFeature['filename']

trainFeature=trainFeature.drop(['filename'],axis=1)
valiFeature=valiFeature.drop(['filename'],axis=1)

In [103]:
from sklearn.decomposition import PCA

pca = PCA(n_components=512)
trainFeature= pca.fit_transform(trainFeature)
valiFeature= pca.transform(valiFeature)

In [104]:
from sklearn.preprocessing import StandardScaler
stdscaler = StandardScaler()
stdscaler.fit(trainFeature)

trainFeature = stdscaler.transform(trainFeature)
valiFeature = stdscaler.transform(valiFeature)

In [105]:
trainLabel= pd.read_csv('/content/gdrive/MyDrive/ml/restnet50/labelCSV/trainLabel.csv')
valiLabel = pd.read_csv('/content/gdrive/MyDrive/ml/restnet50/labelCSV/valiLabel.csv')

trainLabel=trainLabel.drop(['pathname'],axis=1)
valiLabel=valiLabel.drop(['pathname'],axis=1)

In [106]:
trainLabel=pd.get_dummies(trainLabel['label'])
valiLabel=pd.get_dummies(valiLabel['label'])
valiLabel=valiLabel.reindex(columns=trainLabel.columns, fill_value=0)

trainLabel = trainLabel.astype(int)
valiLabel = valiLabel.astype(int)

In [107]:
from tensorflow.keras.optimizers import Adam

In [108]:
model = models.Sequential()
model.add(layers.Dense(2048, input_shape=(512,)))
model.add(layers.BatchNormalization())
model.add(layers.Activation('relu'))

model.add(layers.Dense(4096))
model.add(layers.BatchNormalization())
model.add(layers.Activation('relu'))
model.add(layers.Dropout(0.3))

model.add(layers.Dense(4096))
model.add(layers.BatchNormalization())
model.add(layers.Activation('relu'))
model.add(layers.Dropout(0.3))

model.add(layers.Dense(2048))
model.add(layers.BatchNormalization())
model.add(layers.Activation('relu'))
model.add(layers.Dropout(0.3))

model.add(layers.Dense(1024))
model.add(layers.BatchNormalization())
model.add(layers.Activation('relu'))
model.add(layers.Dropout(0.3))

model.add(layers.Dense(512))
model.add(layers.BatchNormalization())
model.add(layers.Activation('relu'))

model.add(layers.Dense(18, activation='softmax'))
model.summary()

adamm = Adam(learning_rate=0.00005)
model.compile(optimizer=adamm, loss='categorical_crossentropy', metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_29 (Dense)                │ (None, 2048)           │     1,050,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_26          │ (None, 2048)           │         8,192 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_26 (Activation)      │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_30 (Dense)                │ (None, 4096)           │     8,392,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_27          │ (None, 4096)           │        16,384 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_27 (Activation)      │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_15 (Dropout)            │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_31 (Dense)                │ (None, 4096)           │    16,781,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_28          │ (None, 4096)           │        16,384 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_28 (Activation)      │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_16 (Dropout)            │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_32 (Dense)                │ (None, 2048)           │     8,390,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_29          │ (None, 2048)           │         8,192 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_29 (Activation)      │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_17 (Dropout)            │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_33 (Dense)                │ (None, 1024)           │     2,098,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_30          │ (None, 1024)           │         4,096 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_30 (Activation)      │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_18 (Dropout)            │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_34 (Dense)                │ (None, 512)            │       524,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_31          │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_31 (Activation)      │ (None, 512)            │             

 Total params: 37,302,802 (142.30 MB)

 Trainable params: 37,275,154 (142.19 MB)

 Non-trainable params: 27,648 (108.00 KB)

In [109]:
from tensorflow.keras.callbacks import EarlyStopping
early_stop = EarlyStopping(
    monitor='accuracy',
    patience=20,
    restore_best_weights=True
)

In [110]:
history = model.fit(trainFeature, trainLabel, epochs=300,
                    validation_data=(valiFeature, valiLabel),
                    callbacks=[early_stop])

Epoch 1/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 15s 86ms/step - accuracy: 0.0592 - loss: 3.2241 - val_accuracy: 0.0802 - val_loss: 2.8951
Epoch 2/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.0774 - loss: 3.0236 - val_accuracy: 0.0591 - val_loss: 2.9195
Epoch 3/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.1027 - loss: 2.8655 - val_accuracy: 0.0618 - val_loss: 2.9578
Epoch 4/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.1414 - loss: 2.7317 - val_accuracy: 0.0486 - val_loss: 3.0293
Epoch 5/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.1939 - loss: 2.5794 - val_accuracy: 0.0565 - val_loss: 3.0777
Epoch 6/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.2593 - loss: 2.4422 - val_accuracy: 0.0539 - val_loss: 3.1714
Epoch 7/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.2835 - loss: 2.3104 - val_accuracy: 0.0473 - val_loss: 3.2953
Epoch 8/300
90/90 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.3578 - loss: 2.1539 - val_accuracy: 0

In [111]:
new_model = models.Sequential(model.layers[:-1])

for old_layer, new_layer in zip(model.layers[:-1], new_model.layers):
    new_layer.set_weights(old_layer.get_weights())

In [44]:
new_model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_22 (Dense)                │ (None, 2048)           │     1,050,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_20          │ (None, 2048)           │         8,192 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_20 (Activation)      │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 4096)           │     8,392,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_21          │ (None, 4096)           │        16,384 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_21 (Activation)      │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_24 (Dense)                │ (None, 4096)           │    16,781,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_22          │ (None, 4096)           │        16,384 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_22 (Activation)      │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ (None, 2048)           │     8,390,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_23          │ (None, 2048)           │         8,192 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_23 (Activation)      │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ (None, 1024)           │     2,098,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_24          │ (None, 1024)           │         4,096 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_24 (Activation)      │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_27 (Dense)                │ (None, 512)            │       524,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_25          │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_25 (Activation)      │ (None, 512)            │             

 Total params: 37,293,568 (142.26 MB)

 Trainable params: 37,265,920 (142.16 MB)

 Non-trainable params: 27,648 (108.00 KB)

In [112]:
import joblib

filename = 'CNN.sav'
joblib.dump(new_model, filename)

['CNN.sav']

In [113]:
joblib.dump(stdscaler, 'stdScaler.pkl')
joblib.dump(pca, 'pca.pkl')

['pca.pkl']

In [114]:
trainFeature= pd.read_csv('/content/gdrive/MyDrive/ml/restnet50/featureCSV/trainFeature.csv')
valiFeature=pd.read_csv('/content/gdrive/MyDrive/ml/restnet50/featureCSV/valiFeature.csv')

trainFilename=trainFeature['filename']
valiFilename=valiFeature['filename']

trainFeature=trainFeature.drop(['filename'],axis=1)
valiFeature=valiFeature.drop(['filename'],axis=1)

In [115]:
trainFeature= pca.transform(trainFeature)
valiFeature= pca.transform(valiFeature)

In [116]:
trainFeature = new_model.predict(trainFeature, verbose=1)
valiFeature = new_model.predict(valiFeature, verbose=1)

90/90 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step


In [117]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler(feature_range=(-1, 1))
scaler.fit(trainFeature)

trainFeature = scaler.transform(trainFeature)
valiFeature = scaler.transform(valiFeature)

In [118]:
trainFeature = scaler.transform(trainFeature)
valiFeature = scaler.transform(valiFeature)

In [119]:
df_train = pd.DataFrame(trainFeature)
df_val = pd.DataFrame(valiFeature)

df_train.insert(0, "filename", trainFilename)
df_val.insert(0, "filename", valiFilename)

In [120]:
df_train.to_csv("trainVectors.csv", index=False)
df_val.to_csv("valiVectors.csv", index=False)

joblib.dump(scaler, 'minMaxScaler.pkl')

['minMaxScaler.pkl']